# Vertical datums: what does zero metres mean?

**Time:** 40 minutes · **Prerequisites:** chapters 03–04 or basic Python tables. No private input or GIS installation is required.

[Run in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/14_vertical_datums.ipynb)

**Reading the book?** It shows saved results. You can zoom and rotate Plotly figures here; Python-backed selectors, sliders and uploads require running the notebook in JupyterLite.

Wait for **Python (Pyodide) | Idle**, then choose **Run → Run All Cells**. Public results below reflect the bundled aggregate artifact; unavailable models are labeled. Fictional exercises remain explicitly separate from the study.

In [1]:
import sys
from pathlib import Path
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'plotly', 'ipywidgets'])
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from coastlab import show, land_traces
from naturelab import (msl_elevation, screening_summary, paired_comparison,
                       load_public, load_sources, coverage_frame, fictional_samples,
                       aggregate_explorer, sensitivity_controls, terrain_explorer)
public = load_public()
sources = load_sources()
print('Ready: Nature 2026 — public aggregates and explicitly fictional exercises.')
print('Current sampling status:', public['status'])

Ready: Nature 2026 — public aggregates and explicitly fictional exercises.
Current sampling status: partial


## Objectives: distinguish the reference from the quantity

Explain ellipsoid, geoid, mean sea level (MSL) and mean higher high water (MHHW). Check the sign of a datum conversion. Identify when a conversion would be applied twice.

A height needs a reference surface. A benchmark reading of 1 m does not identify the surface called zero, its epoch, or whether the height describes a roof, canopy, terrain or water. The same physical point can have different numeric heights in different reference systems.

| Symbol or term | Meaning in this lesson |
|---|---|
| Ellipsoid | Smooth mathematical reference used for positioning |
| Geoid, N | Gravity-based reference surface, described here by height above the ellipsoid |
| H_EGM | Terrain height relative to an EGM geoid |
| MDT | Mean dynamic topography: mean sea-surface height above its specified geoid |
| H_MSL | Terrain height relative to the mean sea surface |
| MHHW | Mean of daily higher high waters over a tidal reference period; different from MSL |

A geoid is not everywhere identical to the time-averaged ocean surface. The Nature extension asks what a correctly referenced terrain model adds to facility screening. It does not turn a static DEM into a flood forecast.

## Follow the signs through one physical height

First recover a common ellipsoidal height, then subtract the sea surface:

\[
h = H_{EGM} + N_{EGM},\qquad
H_{MSL}=H_{EGM}+N_{EGM}-N_{GOCO06s}-MDT.
\]

These are compatible, colocated inputs in metres, not a recipe for combining arbitrary grids. Holding the terrain and geoids fixed, a larger MDT means a higher mean sea surface and a **smaller** terrain height above it. The example below is invented solely to check that sign.

In [2]:
example = pd.DataFrame({'H_EGM_m': [1.8, 1.8, 1.8],
                        'N_EGM_m': [30.4]*3, 'N_GOCO06s_m': [30.0]*3,
                        'MDT_m': [0.0, 0.4, 0.8]})
example['H_MSL_m'] = msl_elevation(example.H_EGM_m, example.N_EGM_m,
                                  example.N_GOCO06s_m, example.MDT_m)
display(example)
assert np.allclose(example.H_MSL_m, [2.2, 1.8, 1.4])
show(go.Figure(go.Scatter(x=example.MDT_m, y=example.H_MSL_m, mode='lines+markers'))
     .update_layout(title='FICTIONAL sign check: raising the sea lowers relative terrain height',
                    xaxis_title='MDT above GOCO06s (m)', yaxis_title='Terrain above MSL (m)'))

,H_EGM_m,N_EGM_m,N_GOCO06s_m,MDT_m,H_MSL_m
0,1.8,30.4,30.0,0.0,2.2
1,1.8,30.4,30.0,0.4,1.8
2,1.8,30.4,30.0,0.8,1.4


<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="e3f30b8b-9d85-4a4e-9b4c-e6787f97e5be" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("e3f30b8b-9d85-4a4e-9b4c-e6787f97e5be")) { Plotly.newPlot( "e3f30b8b-9d85-4a4e-9b4c-e6787f97e5be", [{"mode":"lines+markers","x":{"dtype":"f8","bdata":"AAAAAAAAAACamZmZmZnZP5qZmZmZmek\u002f"},"y":{"dtype":"f8","bdata":"kJmZmZmZAUC6zMzMzMz8P1NmZmZmZvY\u002f"},"type":"scatter"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"surface":[{"type":"surface","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"mesh3d":[{"type":"mesh3d","colorbar":{"outlinewidth":0,"ticks":""}}],"scatter":[{"fillpattern":{"fillmode":"overlay","size":10,"solidity":0.2},"type":"scatter"}],"parcoords":[{"type":"parcoords","line":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolargl":[{"type":"scatterpolargl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"bar":[{"error_x":{"color":"#2a3f5f"},"error_y":{"color":"#2a3f5f"},"marker":{"line":{"color":"#E5ECF6","width":0.5},"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"bar"}],"scattergeo":[{"type":"scattergeo","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolar":[{"type":"scatterpolar","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"histogram":[{"marker":{"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"histogram"}],"scattergl":[{"type":"scattergl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatter3d":[{"type":"scatter3d","line":{"colorbar":{"outlinewidth":0,"ticks":""}},"marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scattermap":[{"type":"scattermap","marker":{"colorbar":{"

## The input contract matters as much as the formula

HYBRID-CNES-CLS2022 uses GOCO06s and averages 1993–2021. The addendum specifies tide-free geoid calculations, the zero-degree term, and degrees 360 / 2190 / 300 for EGM96 / EGM2008 / GOCO06s. The helper evaluates arithmetic only; it does **not** recreate those geoid fields, interpolation or MDT extrapolation.

The repository consumes authors' already-MSL rasters. Applying the equation again would double-correct them. The historical workbook's `ACTUALE` is modeled as `ACTUAL_MHHW_ELEV_M`; it is not established as an EGM-referenced input. Subtracting MDT from it without the tidal-reference relationship is unjustified. An MHHW-to-MSL relationship must be documented for the place and reference period.

## Lab: diagnose three workflows

1. An analyst has H_EGM=1.8, N_EGM=30.4, N_GOCO06s=30.0 and MDT=0.8 m. Compute the result and explain each sign.
2. Another analyst receives a TIFF explicitly referenced to MSL and subtracts MDT again. Explain the error.
3. A report calls its original height MHHW and a new height MSL. What extra evidence is necessary before interpreting their difference as a datum change?

**Worked answers:** (1) 1.4 m: recover ellipsoidal terrain and subtract the mean sea surface. (2) The sea-level adjustment was already included. (3) Obtain the local tidal-datum relation and epochs; also investigate terrain source, resolution and acquisition time. Their difference alone cannot isolate datum effects.

**Deliverable:** A four-line provenance label stating quantity, unit, reference surface and epoch for each input. Write “unknown” wherever documentation is absent.

## Sources and evidence boundaries

Seeger & Minderhoud (2026), [*Sea level much higher than assumed in most coastal hazard assessments*](https://doi.org/10.1038/s41586-026-10196-1), with the [25 August 2026 addendum](https://doi.org/10.1038/s41586-026-11017-1). The pinned [October 2026 catalog, record 23341921](https://doi.org/10.5281/zenodo.23341921) describes corrected products and their licenses. Exact versions, URLs, roles and inspected metadata hashes are in `data/nature-sources.json`; current results are in `data/nature-aggregates.json`.

Original calculations/code here are independent MIT implementations. Corrected-raster derivatives are **CC BY-NC-SA 4.0**, distinct from the **CC BY 4.0** original lesson prose and paper. Cite the corrected product, underlying DEM and MDT/geoid sources. No private workbook, facility identifiers, names, coordinates or clinical counts are bundled. [Full source audit](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/source-audit.html) · [Data dictionary](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/data-dictionary.html) · [References](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/book/references.html).

In [3]:
print('Nature lesson complete: 14_vertical_datums')

Nature lesson complete: 14_vertical_datums
